In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, time

REPO = Path.cwd().resolve()
if not (REPO / "scripts" / "prepare_pusht_demo.py").is_file() or not (REPO / "src" / "sample" / "rollout.py").is_file():
    raise RuntimeError("Open this notebook with the project repository as the working directory.")
DEMO_HOME = Path(os.environ.get("NANOWM_DEMO_HOME", str(REPO / "nanowm-pusht"))).expanduser().resolve()
NUM_SAMPLES = 3
SAMPLING_STEPS = 50  # 15: quick preview; 50: default; 250: slower comparison, no quality guarantee
SEED = 42
BATCH_SIZE = 1  # keeps the memory footprint small
DEMO_HOME.mkdir(parents=True, exist_ok=True)
ASSETS = DEMO_HOME / "assets"
OUTPUT = Path(os.environ.get("WM_ROLLOUTS_DIR", REPO.parent / "outputs/wm-rollouts")) / "pusht-small_policy"

def run(argv, **kwargs):
    print("Running:", " ".join(map(str, argv)), flush=True)
    subprocess.run(list(map(str, argv)), check=True, **kwargs)

run([sys.executable, "-c", "import torch; assert torch.cuda.is_available(), 'CUDA GPU is not available in the current Python environment'; print(torch.cuda.get_device_name(0), torch.version.cuda)"])
print("Project:", REPO)
print("Workspace:", DEMO_HOME)


In [ ]:
REPO

## Load the model and the validation dataset


In [ ]:
#run([sys.executable,"-Xfrozen_modules=off","scripts/prepare_pusht_demo.py", "--root", str(ASSETS), "--num-samples", str(NUM_SAMPLES)], cwd=REPO)
from scripts.prepare_pusht import prepare

prepare(Path(ASSETS),NUM_SAMPLES)

provenance = json.loads((ASSETS / "provenance.json").read_text())


## Generate predictions


In [ ]:
started = time.monotonic()
run([sys.executable,
     "src/sample/rollout_pusht.py",
     "--config", provenance["config"], "--ckpt", provenance["checkpoint"],
     "--save_path", str(OUTPUT), "--num_samples", str(NUM_SAMPLES), "--batch_size", str(BATCH_SIZE),
     "--rollout_length", "20", "--history_length", "3",
     "--num_sampling_steps", str(SAMPLING_STEPS), "--scheduling_mode", "sequential",
     "--history_stabilization_level", "0.02", "--fps", "8",
     "--seed", str(SEED), "--unique_trajectories"], cwd=REPO)
report = json.loads((OUTPUT / "report.json").read_text())
assert len(report["samples"]) == NUM_SAMPLES
print(f"Finished in {time.monotonic()-started:.1f} seconds; peak model GPU allocation {report['peak_gpu_memory_gib']:.2f} GiB.")


## 4. Watch every result

**Left = recorded validation images. Right = model prediction.** The first 3 frames are context.
Play each video or scrub to the end; do not judge the moving agent from the mostly static background.
Metrics below cover only the 17 predicted frames, before video compression. The frozen-frame baseline
illustrates why high full-image PSNR/SSIM can be misleading in this small, mostly static scene.

In [ ]:
from IPython.display import display, Video, Markdown
for sample in report["samples"]:
    i = sample["sample_id"]
    display(Markdown(f"### Clip {i+1} · validation episode {sample['traj_idx']}\nRecorded (left) | predicted (right); 3 context frames + 17 predicted frames."))
    display(Video(filename=str(OUTPUT / f"sample_{i:04d}_compare.mp4"), embed=True, width=768, html_attributes='controls loop muted playsinline style="max-width:100%;height:auto"'))
rows = ["| Clip | Prediction PSNR / SSIM | Frozen last context PSNR / SSIM |", "|---|---|---|"]
for s in report["samples"]:
    a, b = s["prediction"], s["repeat_last_context"]
    rows.append(f"| {s['sample_id']+1} | {a['psnr_db']:.2f} dB / {a['ssim']:.4f} | {b['psnr_db']:.2f} dB / {b['ssim']:.4f} |")
display(Markdown("\n".join(rows)))

## Evaluate the checkpoint

In [ ]:
run([sys.executable, "src/sample/evaluate_metrics.py",
    "--video_dir", str(OUTPUT),
    "--history_length", "3" ,
    "--output_csv", "metrics.csv"], cwd=REPO)